In [13]:
import pandas as pd
import plotly.graph_objects as go

In [14]:
CSV_PATH = "../data/institution_network_evolution_LHC_2000_2020.csv"
HTML_PATH = "../figures/LHC_2.html"

SOURCE_LAT_COL = "source_lat"
SOURCE_LON_COL = "source_lng"
DEST_LAT_COL   = "target_lat"
DEST_LON_COL   = "target_lng"
YEAR_COL       = "publication_year"

MAP_HEIGHT = 800
LINE_WIDTH = 1.5
POINT_SIZE = 7

In [15]:
df = pd.read_csv(CSV_PATH)

required_cols = [
    SOURCE_LAT_COL, SOURCE_LON_COL,
    DEST_LAT_COL, DEST_LON_COL,
    YEAR_COL
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# Clean data
df = df.dropna(subset=required_cols).copy()
df[YEAR_COL] = df[YEAR_COL].astype(str)

# Optional: ensure numeric lat/lon
for col in [SOURCE_LAT_COL, SOURCE_LON_COL, DEST_LAT_COL, DEST_LON_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=[SOURCE_LAT_COL, SOURCE_LON_COL, DEST_LAT_COL, DEST_LON_COL])

In [16]:
def make_edge_trace(dataframe, name, visible=True):
    lats = []
    lons = []
    hover_text = []

    for _, row in dataframe.iterrows():
        lats.extend([row[SOURCE_LAT_COL], row[DEST_LAT_COL], None])
        lons.extend([row[SOURCE_LON_COL], row[DEST_LON_COL], None])
        hover_text.extend([
            f"Year: {row[YEAR_COL]}<br>Source: ({row[SOURCE_LAT_COL]}, {row[SOURCE_LON_COL]})<br>Dest: ({row[DEST_LAT_COL]}, {row[DEST_LON_COL]})",
            f"Year: {row[YEAR_COL]}<br>Source: ({row[SOURCE_LAT_COL]}, {row[SOURCE_LON_COL]})<br>Dest: ({row[DEST_LAT_COL]}, {row[DEST_LON_COL]})",
            None
        ])

    return go.Scattermap(
        lat=lats,
        lon=lons,
        mode="lines",
        line=dict(width=LINE_WIDTH),
        name=name,
        visible=visible,
        hoverinfo="text",
        text=hover_text
    )

def make_point_trace(dataframe, name, visible=True):
    src_points = dataframe[[SOURCE_LAT_COL, SOURCE_LON_COL, YEAR_COL]].rename(
        columns={SOURCE_LAT_COL: "lat", SOURCE_LON_COL: "lon"}
    )
    src_points["point_type"] = "source"

    dst_points = dataframe[[DEST_LAT_COL, DEST_LON_COL, YEAR_COL]].rename(
        columns={DEST_LAT_COL: "lat", DEST_LON_COL: "lon"}
    )
    dst_points["point_type"] = "destination"

    points = pd.concat([src_points, dst_points], ignore_index=True)

    return go.Scattermap(
        lat=points["lat"],
        lon=points["lon"],
        mode="markers",
        marker=dict(size=POINT_SIZE),
        name=name,
        visible=visible,
        hoverinfo="text",
        text=[
            f"Year: {y}<br>Type: {t}<br>Point: ({lat}, {lon})"
            for lat, lon, y, t in zip(
                points["lat"], points["lon"], points[YEAR_COL], points["point_type"]
            )
        ]
    )

In [17]:
fig = go.Figure()

years = sorted(df[YEAR_COL].unique(), key=lambda x: (len(x), x))

# Add "All years" traces first
fig.add_trace(make_edge_trace(df, "Edges - All years", visible=True))
fig.add_trace(make_point_trace(df, "Points - All years", visible=True))

# Add one pair of traces per year
for year in years:
    dfi = df[df[YEAR_COL] == year]
    fig.add_trace(make_edge_trace(dfi, f"Edges - {year}", visible=False))
    fig.add_trace(make_point_trace(dfi, f"Points - {year}", visible=False))

In [18]:
buttons = []

# Button for all years
visible_all = [False] * len(fig.data)
visible_all[0] = True
visible_all[1] = True

buttons.append(
    dict(
        label="All years",
        method="update",
        args=[
            {"visible": visible_all},
            {"title": "Interactive Source-Destination Map - All years"}
        ]
    )
)

# Buttons for each year
for i, year in enumerate(years):
    visible = [False] * len(fig.data)
    edge_idx = 2 + i * 2
    point_idx = 3 + i * 2
    visible[edge_idx] = True
    visible[point_idx] = True

    buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[
                {"visible": visible},
                {"title": f"Interactive Source-Destination Map - {year}"}
            ]
        )
    )

In [19]:
all_lats = pd.concat([df[SOURCE_LAT_COL], df[DEST_LAT_COL]], ignore_index=True)
all_lons = pd.concat([df[SOURCE_LON_COL], df[DEST_LON_COL]], ignore_index=True)

center_lat = all_lats.mean()
center_lon = all_lons.mean()

In [20]:
fig.update_layout(
    title="Interactive Source-Destination Map - All years",
    height=MAP_HEIGHT,
    map=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=2
    ),
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01,
            y=0.99,
            xanchor="left",
            yanchor="top",
            showactive=True,
            buttons=buttons
        )
    ],
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="left",
        x=0.01
    )
)

In [21]:
fig.write_html(HTML_PATH)